# Reproduce Identity2Vec (I2V) — 3-seed evaluation

Run top to bottom (**Shift+Enter**). You edit **two lines** in Step 1: `DATASET` and `SEEDS`.

The pipeline: pick a dataset (auto-aligned to a safe graph+labels version if needed) → train a fresh embedding **per seed** → score **node classification** (micro/macro/weighted F1) and **link prediction** (AUC) → report **mean ± std** across seeds.

All heavy logic lives in the project files (`make_labels.py`, `scripts/runner.py`); the notebook only sets knobs and calls them.


## Step 0 — Set up

Tells Python where the project is. Run once.


In [1]:
import os, sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
os.chdir(ROOT)
for p in (str(ROOT), str(ROOT / 'scripts')):
    if p not in sys.path:
        sys.path.insert(0, p)
print('Working from:', Path.cwd())

Working from: /home/m-adam/identity2vec


## Step 1 — Choose dataset + seeds ⬅️ the only place you edit

`prepare_dataset` resolves the dataset to a **safe** version (e.g. author `citeseer` → aligned `citeseer_linqs`, built from one LINQS source so graph node-ids and labels match). It prints what it used and why, and never touches the original author files.


In [2]:
DATASET = "enzymes"        # cora | citeseer_linqs | pubmed | pubmed_linqs | amazon_computers | amazon_photo | coauthor_cs | coauthor_physics
SEEDS   = [42, 43, 44]      # 3 seeds for mean ± std
    
from make_labels import prepare_dataset
info = prepare_dataset(DATASET)   # -> {base, version, safe, edge_path, label_path}; builds aligned version if missing

dataset ready -> base=enzymes version=orig safe=enzymes
  edges  = input/enzymes.edgelist
  labels = labels/enzymes.labels


## Step 2 — Node classification (one embedding per seed)

For each seed: train a full-graph embedding → 70/30 stratified label split → logistic regression → **micro / macro / weighted F1**.
Embeddings are saved as `output/notebook1_reproduce_i2v/{dataset}/node_classification/{model}_s{seed}.emb` — the folder names the notebook, dataset, and task; the file names the model and seed.


In [ ]:
from runner import run_nodeclass_repeated
node_rows = run_nodeclass_repeated(info, seeds=SEEDS)

STARTING RANDOM WALK... 
Number of Nodes: 19474


Current Walk: 1 of 10


  0%|          | 63/19474 [00:25<2:07:18,  2.54it/s]

## Step 3 — Link prediction (one split + embedding per seed)

For each seed: hide 30% of edges → train an embedding on the **70% train graph only** (leakage-free) → **AUC**.
Splits: `splits/link_prediction/original_graph/{dataset}/seed_{seed}/` (4 files: `train.edgelist`, `train_neg.txt`, `test_pos.txt`, `test_neg.txt`); embeddings: `output/notebook1_reproduce_i2v/{dataset}/link_prediction/{model}_s{seed}.emb`.


In [ ]:
from runner import run_linkpred_repeated
lp_rows = run_linkpred_repeated(info, seeds=SEEDS)

STARTING RANDOM WALK... 
Number of Nodes: 2485


Current Walk: 1 of 10


  0%|          | 6/2485 [00:00<00:48, 51.49it/s]



Current Walk: 2 of 10


  0%|          | 5/2485 [00:00<00:58, 42.23it/s]



Current Walk: 3 of 10


...



Current Walk: 9 of 10


  0%|          | 0/2485 [00:00<?, ?it/s]



Current Walk: 10 of 10


100%|██████████| 2485/2485 [00:58<00:00, 42.48it/s]


Training Node Corpus...
Saving Embeddings...
  [identity2vec lp s42] AUC=0.8143 (logreg=0.8163 cosine=0.8143)


## Step 4 — Results + summary (mean ± std)

Per-seed table, then the summary (`mean`, sample `std`, `delta = max − min`). Both saved to `results/`.


In [ ]:
import os
from runner import summarize_seed_results

per_seed, summary = summarize_seed_results(node_rows, lp_rows)

tag = f"{info['base']}_{info['version']}"
res_dir = f"results/notebook1_reproduce_i2v/{info['safe']}"   # Phase-1 result tables live in the notebook-1 zone
os.makedirs(res_dir, exist_ok=True)
per_seed.to_csv(f"{res_dir}/{tag}_per_seed.csv", index=False)
summary.to_csv(f"{res_dir}/{tag}_summary.csv", index=False)

def make_clean_table(per_seed, summary, task):
    task_rows = per_seed[per_seed["task"] == task].copy()

    if task == "nodeclass":
        metrics = ["micro_f1", "macro_f1", "weighted_f1"]
    elif task == "linkpred":
        metrics = ["auc"]
    else:
        raise ValueError(f"Unknown task: {task}")

    long_rows = task_rows.melt(
        id_vars=["dataset", "version", "task", "seed"],
        value_vars=metrics,
        var_name="metric",
        value_name="score",
    ).dropna(subset=["score"])

    seed_table = long_rows.pivot_table(
        index=["dataset", "version", "task", "metric"],
        columns="seed",
        values="score",
        aggfunc="first",
    ).reset_index()

    seed_table = seed_table.rename(columns={
        42: "seed_42",
        43: "seed_43",
        44: "seed_44",
    })

    summary_task = summary[summary["task"] == task].copy()

    clean = seed_table.merge(
        summary_task[["dataset", "version", "task", "metric", "mean", "std", "delta"]],
        on=["dataset", "version", "task", "metric"],
        how="left",
    )

    clean["final_score"] = clean.apply(
        lambda r: f"{r['mean']:.4f} ± {r['std']:.4f}",
        axis=1,
    )

    # metric 1st column, task 2nd column on both tables; rows numbered from 1 not 0
    lead = ["metric", "task"]
    clean = clean[lead + [c for c in clean.columns if c not in lead]]
    clean = clean.sort_values("metric").reset_index(drop=True)
    clean.index = range(1, len(clean) + 1)

    return clean

node_table = make_clean_table(per_seed, summary, "nodeclass")
lp_table = make_clean_table(per_seed, summary, "linkpred")

node_table.to_csv(f"{res_dir}/{tag}_nodeclass_table.csv", index=False)
lp_table.to_csv(f"{res_dir}/{tag}_linkpred_table.csv", index=False)

print("\n=== Node Classification Table ===")
display(node_table)

print("\n=== Link Prediction Table ===")
display(lp_table)

## Step 5 — Cross-model benchmark (I2V vs deepwalk / node2vec / struc2vec)

**Edit only the 3 lists at the top of the cell below** (`RUN_DATASETS`, `RUN_MODELS`, `RUN_SEEDS`), then run it. **Do NOT change Step 1's `DATASET`** — that variable is only for the single-dataset Steps 2–4.

Examples:

- Cora + Identity2Vec: `RUN_DATASETS=["cora"]`, `RUN_MODELS=["identity2vec"]`, `RUN_SEEDS=[42]`
- same dataset, DeepWalk: change one line → `RUN_MODELS=["deepwalk"]`
- full comparison: `RUN_DATASETS=["cora","citeseer_linqs","enzymes","webkb_wisc"]`, `RUN_MODELS=["identity2vec","deepwalk","node2vec","struc2vec"]`, `RUN_SEEDS=[42,43,44]`

Outputs two tables (rows = datasets, columns = methods): **Table 1** node-class weighted F1 (mean ± std), **Table 2** link-pred AUC (mean ± std). Embeddings + splits are cached by filename, so re-runs train only what's missing. Saved under `results/notebook1_reproduce_i2v/benchmark/`. _(politics was dropped — rt-pol has no verifiable node labels; webkb_wisc replaces it, fully labelled with 5 classes.)_


In [ ]:
# # Step 5 — Cross-model benchmark.
# # EDIT ONLY these 3 lists to choose what runs.

# RUN_DATASETS = ["cora"]
# RUN_MODELS   = ["identity2vec"]
# RUN_SEEDS    = [42]

# from benchmark_baselines import run_benchmark, save_benchmark


# def clean_table_for_display(table, dataset_order=None, model_order=None):
#     """Remove Pandas index/column names and show Dataset as a normal column."""
#     table = table.copy()

#     if table.empty:
#         return table

#     table.index.name = None
#     table.columns.name = None

#     table = table.reset_index()
#     table = table.rename(columns={table.columns[0]: "Dataset"})

#     if dataset_order is not None and "Dataset" in table.columns:
#         table["_dataset_order"] = table["Dataset"].apply(
#             lambda x: dataset_order.index(x) if x in dataset_order else len(dataset_order)
#         )
#         table = table.sort_values("_dataset_order").drop(columns=["_dataset_order"])

#     if model_order is not None:
#         ordered_cols = ["Dataset"]
#         ordered_cols += [m for m in model_order if m in table.columns]
#         ordered_cols += [c for c in table.columns if c not in ordered_cols]
#         table = table[ordered_cols]

#     return table.reset_index(drop=True)


# # Show per-seed training progress, then the final tables.
# bench_rows = run_benchmark(
#     datasets=RUN_DATASETS,
#     models=RUN_MODELS,
#     seeds=RUN_SEEDS,
# )

# nc_table, lp_table = save_benchmark(bench_rows)


# nc_show = clean_table_for_display(
#     nc_table,
#     dataset_order=RUN_DATASETS,
#     model_order=RUN_MODELS,
# )

# lp_show = clean_table_for_display(
#     lp_table,
#     dataset_order=RUN_DATASETS,
#     model_order=RUN_MODELS,
# )

# print("=== Table 1: Node Classification (Weighted F1, mean ± std) ===")
# display(nc_show)

# print("=== Table 2: Link Prediction (AUC, mean ± std) ===")
# display(lp_show)

Label file already exists: labels/cora.labels
dataset ready -> base=cora version=orig safe=cora
  edges  = input/cora.edgelist
  labels = labels/cora.labels

=== identity2vec on cora ===
  [identity2vec nc s42] reuse existing cora_nc_orig_s42.emb
  [identity2vec nc s42] micro=0.7183 macro=0.7038 weighted=0.7160
  [identity2vec lp s42] reuse existing cora_lp_orig_s42.emb
  [identity2vec lp s42] AUC=0.8143 (logreg=0.8163 cosine=0.8143)
=== Table 1: Node Classification (Weighted F1, mean ± std) ===


,Dataset,identity2vec
0,cora,0.7160 ± 0.0000


=== Table 2: Link Prediction (AUC, mean ± std) ===


,Dataset,identity2vec
0,cora,0.8143 ± 0.0000


In [ ]:
# import pandas as pd  # add if not already imported in this cell

# # ============================================================
# # Original I2V (paper) reference results — for side-by-side comparison.
# # Column order = RUN_MODELS, so these line up with your reproduced tables.
# # Paper reports single numbers (no std); "____" = fill in yourself.
# # ============================================================

# # Link Prediction — paper Table 4 (AUC). Cora filled from the table you sent.
# PAPER_LP = {
#     "cora": {"identity2vec": 0.8413, "deepwalk": 0.7529, "node2vec": 0.7658, "struc2vec": 0.7115},
#     # Other paper Table 4 rows (uncomment when you add them to RUN_DATASETS):
#     # "citeseer_linqs": {"identity2vec": 0.8373, "deepwalk": 0.7301, "node2vec": 0.7951, "struc2vec": 0.6962},  # NOTE: paper's Citeseer is a DIFFERENT graph than citeseer_linqs
#     # "enzymes":        {"identity2vec": 0.8024, "deepwalk": 0.7248, "node2vec": 0.7419, "struc2vec": 0.6902},
#     # webkb_wisc: NOT in the paper's Table 4 (webkb only appears in Figure 2) -> stays "____".
#     # LINE (paper, not in your models): Cora = 0.5407, if you ever add it.
# }

# # Node Classification — paper reports a PLOT (Figure 5, F1 vs train-ratio), not a table.
# # Read values off the figure at your train ratio and fill these in.
# PAPER_NC = {
#     "cora": {"identity2vec": 0.7345, "deepwalk": 0.6162, "node2vec": 0.6384, "struc2vec": 0.6912},
# }

# # Build a paper table in the same Dataset x models shape ("____" where unfilled).
# def paper_table(values, datasets, models):
#     """Turn the manual {dataset: {model: value}} dict into a display table."""
#     def cell(d, m):
#         v = values.get(d, {}).get(m)
#         return f"{v:.4f}" if isinstance(v, (int, float)) else "____"
#     rows = [{"Dataset": d, **{m: cell(d, m) for m in models}} for d in datasets]
#     return pd.DataFrame(rows)[["Dataset"] + models]

# print("=== Table 1: Node Classification (Weighted F1, mean ± std) ===")
# display(nc_show)

# print("=== Original I2V Results — Node Classification (paper, Figure 5) ===")
# display(paper_table(PAPER_NC, RUN_DATASETS, RUN_MODELS))

# print("=== Table 2: Link Prediction (AUC, mean ± std) ===")
# display(lp_show)

# print("=== Original I2V Results — Link Prediction (paper, Table 4) ===")
# display(paper_table(PAPER_LP, RUN_DATASETS, RUN_MODELS))

=== Table 1: Node Classification (Weighted F1, mean ± std) ===


,Dataset,identity2vec
0,cora,0.7160 ± 0.0000


=== Original I2V Results — Node Classification (paper, Figure 5) ===


,Dataset,identity2vec
0,cora,0.7345


=== Table 2: Link Prediction (AUC, mean ± std) ===


,Dataset,identity2vec
0,cora,0.8143 ± 0.0000


=== Original I2V Results — Link Prediction (paper, Table 4) ===


,Dataset,identity2vec
0,cora,0.8413


In [ ]:
# ## Step 6 — Dataset-wise comparison from saved results + existing files only
# # This cell does NOT train anything.
# # It shows only one final table.
# # It checks:
# # 1) results/notebook1_reproduce_i2v/benchmark/benchmark_per_seed.csv
# # 2) old Steps 2–4 I2V result files
# # 3) output/ and splits/ files already present on disk

# from pathlib import Path
# from contextlib import redirect_stdout
# import io
# import pandas as pd

# from make_labels import prepare_dataset


# COMPARE_DATASETS = ["cora", "citeseer_linqs"]   # example: ["cora", "citeseer_linqs", "enzymes", "webkb_wisc"]
# COMPARE_MODELS   = ["identity2vec", "deepwalk", "node2vec", "struc2vec"]
# COMPARE_SEEDS    = [42, 43, 44]

# NB1_OUT = Path("output") / "notebook1_reproduce_i2v"                       # embeddings zone
# NB1_RES = Path("results") / "notebook1_reproduce_i2v"                      # result tables zone
# LP_SPLITS = Path("splits") / "link_prediction" / "original_graph"          # per-seed split folders


# def quiet_prepare_dataset(dataset):
#     silent = io.StringIO()
#     with redirect_stdout(silent):
#         info = prepare_dataset(dataset)
#     return info


# def model_short(model):
#     """Filename form of a model: identity2vec -> i2v, others keep their name."""
#     return "i2v" if model == "identity2vec" else model


# def embedding_exists(info, model, task_folder, seed):
#     return (NB1_OUT / info["safe"] / task_folder / f"{model_short(model)}_s{seed}.emb").exists()


# def split_exists(info, seed):
#     d = LP_SPLITS / info["safe"] / f"seed_{seed}"
#     return (d / "train.edgelist").exists() and (d / "test_pos.txt").exists() and (d / "test_neg.txt").exists()


# def normalize_results_columns(df):
#     df = df.copy()

#     if "method" in df.columns and "model" not in df.columns:
#         df = df.rename(columns={"method": "model"})

#     if "dataset_name" in df.columns and "dataset" not in df.columns:
#         df = df.rename(columns={"dataset_name": "dataset"})

#     if "weighted" in df.columns and "weighted_f1" not in df.columns:
#         df = df.rename(columns={"weighted": "weighted_f1"})

#     if "roc_auc" in df.columns and "auc" not in df.columns:
#         df = df.rename(columns={"roc_auc": "auc"})

#     if "task" in df.columns:
#         df["task"] = df["task"].replace({
#             "nc": "nodeclass",
#             "node_classification": "nodeclass",
#             "nodeclassification": "nodeclass",
#             "lp": "linkpred",
#             "link_prediction": "linkpred",
#             "linkprediction": "linkpred",
#         })

#     return df


# def load_saved_results(datasets):
#     frames = []

#     bench_file = NB1_RES / "benchmark" / "benchmark_per_seed.csv"
#     if bench_file.exists():
#         df = pd.read_csv(bench_file)
#         df = normalize_results_columns(df)
#         frames.append(df)

#     for dataset in datasets:
#         info = quiet_prepare_dataset(dataset)

#         old_file = NB1_RES / info["safe"] / f"{info['base']}_{info['version']}_per_seed.csv"

#         if old_file.exists():
#             df = pd.read_csv(old_file)
#             df = normalize_results_columns(df)

#             # Old Steps 2–4 are Identity2Vec-only.
#             df["dataset"] = info["safe"]
#             df["model"] = "identity2vec"

#             frames.append(df)

#     if len(frames) == 0:
#         return pd.DataFrame()

#     results = pd.concat(frames, ignore_index=True)
#     results = normalize_results_columns(results)

#     needed = ["dataset", "model", "task", "seed"]
#     if all(col in results.columns for col in needed):
#         results = results.drop_duplicates(
#             subset=["dataset", "model", "task", "seed"],
#             keep="last",
#         )

#     return results


# def saved_score_or_none(results, dataset_names, model, task, metric):
#     if results.empty:
#         return None

#     required_cols = ["dataset", "model", "task", "seed", metric]
#     if not all(col in results.columns for col in required_cols):
#         return None

#     rows = results[
#         (results["dataset"].isin(dataset_names))
#         & (results["model"] == model)
#         & (results["task"] == task)
#         & (results["seed"].isin(COMPARE_SEEDS))
#         & (results[metric].notna())
#     ].copy()

#     done_seeds = sorted(rows["seed"].unique().tolist())

#     if len(done_seeds) == 0:
#         return None

#     mean = rows[metric].mean()
#     std = rows[metric].std(ddof=1) if len(rows) > 1 else 0.0

#     if len(done_seeds) < len(COMPARE_SEEDS):
#         return f"{mean:.4f} ± {std:.4f} ({len(done_seeds)}/{len(COMPARE_SEEDS)} seeds)"

#     return f"{mean:.4f} ± {std:.4f}"


# def file_status(info, model, task):
#     if task == "nodeclass":
#         done = [
#             seed for seed in COMPARE_SEEDS
#             if embedding_exists(info, model, "node_classification", seed)
#         ]

#         if len(done) == len(COMPARE_SEEDS):
#             return "FILES READY, SCORE MISSING"

#         return f"WAIT: NC embeddings {len(done)}/{len(COMPARE_SEEDS)}"

#     if task == "linkpred":
#         done = [
#             seed for seed in COMPARE_SEEDS
#             if embedding_exists(info, model, "link_prediction", seed) and split_exists(info, seed)
#         ]

#         if len(done) == len(COMPARE_SEEDS):
#             return "FILES READY, SCORE MISSING"

#         return f"WAIT: LP files {len(done)}/{len(COMPARE_SEEDS)}"

#     raise ValueError(f"Unknown task: {task}")


# results = load_saved_results(COMPARE_DATASETS)

# all_rows = []

# for dataset in COMPARE_DATASETS:
#     info = quiet_prepare_dataset(dataset)

#     dataset_names = list({
#         dataset,
#         info["safe"],
#         info["base"],
#     })

#     for model in COMPARE_MODELS:
#         nc_score = saved_score_or_none(
#             results=results,
#             dataset_names=dataset_names,
#             model=model,
#             task="nodeclass",
#             metric="weighted_f1",
#         )

#         lp_score = saved_score_or_none(
#             results=results,
#             dataset_names=dataset_names,
#             model=model,
#             task="linkpred",
#             metric="auc",
#         )

#         all_rows.append({
#             "Dataset": info["safe"],
#             "Model": model,
#             "NC weighted F1": nc_score if nc_score is not None else file_status(info, model, "nodeclass"),
#             "LP AUC": lp_score if lp_score is not None else file_status(info, model, "linkpred"),
#         })

# compare_table = pd.DataFrame(all_rows)

# display(compare_table)